# Verne, small, step 08: the instruction, on T5

No explained notebook yet. Step 07 (same day, T1-wide): `07_low_grasp.ipynb`.

T5 (`instructed_pick_place`): two cubes of two of 4 colours on a near and a far spot (14 cm apart, along the robot's forward axis), two grey bins left and right, and "put the green cube in the left box". A policy that can't read is at chance: 1/2 on the cube, 1/2 on the bin. Before Verne: right cube 42-45% for SmolVLA, 49% for our flow matching with a one-hot.

**Demos:** `instructed_pick_place_square_noise005_n150`: the T5 expert now squares up to the *named* cube (T2's expert reuses T1's step; `cube_yaw` reads the named cube) and grasps low (step 07), with the full shake; expert 91% (150 of 165). Split 135 / 15.

**Instruction:** a one-hot of 6 words (red, green, blue, yellow, left, right) through Linear(6 → 256) as one more video token (`data.WORDS`, `model.instruction_in`); T1 frames read all zeros. Runs on Modal (L40S), seed 0, 50 rollout seeds, failures split into `wrong_cube` / `wrong_bin`.

### step 08.1 — the VAE had to change first

The step 03 VAE (trained on T1's red cube and blue bin only) painted every T5 cube red and the grey bins blue. `experiments/verne/scripts/vae_colour_check.py`, on T5's 15 held-out demos (187 frames), cube pixels that come back the same colour:

```-
VAE                     recon     red    green   blue   yellow
step 03 (T1 only)       0.0307    84%     7%      0%     82%   (yellow: a pale red blob, nearest to yellow)
step 07 (all tasks)     0.0020    98%    98%     99%     97%
```

The new VAE (`run vae 07_instruction`): T1, T1-wide low, T1-wide ±45° bias, T5 frames, every 2nd frame (49k). Two attempts first: cosine on the local 4060 reached 0.0015 and then crashed computing the latent stats before saving (no checkpoints then); WSD at lr 1e-3 on a pod spiked 20× at steps 3500 and 7000. Then WSD at 5e-4 with gradient clipping and kept checkpoints: it spiked 5× at step 7000 too (same seed, same batches), so its `step6000.pt` (recon 0.0017) became the VAE (`experiments/verne/scripts/vae_from_ckpt.py`).

In [ ]:
import json

from dream_robot.policies.verne.run import RUNS

RUNS_08 = {
    'instruction on the video side': RUNS / '07_instruction' / 't5' / 'seed0',
    '+ added to the action tokens': RUNS / '07_instruction_act' / 't5' / 'seed0',
}

In [ ]:
def read_run(path):
    # a run folder -> n_exec -> (successes, failure counts), probe fit
    out = {}
    for n in (8, 16):
        res = json.loads((path / f'sweep_{n}.json').read_text()).values()
        fails = {}
        for ok, mode, _ in res:
            if not ok:
                fails[mode] = fails.get(mode, 0) + 1
        out[n] = (sum(r[0] for r in res), fails)
    return out, json.loads((path / 'probe.json').read_text())['fit']

### step 08.2 — scores

In [ ]:
print(f'{"":32s} {"n_exec 8":>9s} {"n_exec 16":>10s}  failures (n_exec 8 | 16)')
for name, path in RUNS_08.items():
    r, fit = read_run(path)
    print(f'{name:32s} {r[8][0]:6d}/50 {r[16][0]:7d}/50   {r[8][1]} | {r[16][1]}')

                                  n_exec 8  n_exec 16  failures (n_exec 8 | 16)
instruction on the video side        22/50      22/50   {'wrong_cube': 21, 'no_grasp': 4, 'dropped': 3} | {'wrong_cube': 22, 'no_grasp': 5, 'dropped': 1}
+ added to the action tokens         21/50      17/50   {'wrong_cube': 23, 'no_grasp': 5, 'dropped': 1} | {'wrong_cube': 23, 'no_grasp': 8, 'dropped': 2}


**22/50, and the bin is always right: 0 `wrong_bin`. Almost every failure is `wrong_cube`.**
- The side word works: "left/right box" always means the same place.
- The colour word doesn't: the yellow cube is near in some episodes and far in others, so the policy has to find it in the image.
- Adding the instruction to the action tokens too (Verne's action blocks read the text directly) changed nothing: 21/17.
- Films (first run): [placed, green → left](../../scratch/verne07/t5_seed1001_green_left_placed.mp4), [wrong cube: yellow asked, took green](../../scratch/verne07/t5_seed1000_yellow_right_wrong_cube.mp4), [took red](../../scratch/verne07/t5_seed1002_yellow_right_wrong_cube.mp4). Motion, square-up, grasp and bin are confident; it commits to one cube from the first chunk.

### step 08.3 — which cube it takes

`experiments/verne/scripts/t5_pick_rule.py` (the cube nearest the first close, against the env's draw on the same seeds):

```-
run                          named cube   what it learned instead
video side                   25/50        "go for red": named red 16/19, blue 1/9; red taken 26 of 50
+ action side                22/50        "go far": the far spot taken 31 of 50; red 9/17
```

Each run settles on its own arbitrary default; the colour word isn't what decides.

`experiments/verne/scripts/t5_instruction_probe.py` (first run, same noise per episode): swapping the colour word (side kept) changes the cube in 12/50 episodes, zeroing it in 7/50; the first planned chunk moves 0.023 rad on a swap against 0.016 from sampling noise alone. The word reaches the actions, weakly.

### step 08.4 — is the binding learnable from these latents at all?

`experiments/verne/scripts/t5_colour_binding.py` (detail in `pending_walkthroughs.md`): a small conv net gets only the question "picture + colour word → is that cube on the near or the far spot?", on fresh T5 scenes (first frames, through the step 07 VAE). Chance 50%.

```-
training scenes     train    held-out
135 (demo count)    100%     89%
1,000               100%     100%
```

**Easy.** The latents carry colour and place; the binding is learnable from 135 scenes and solved by 1,000. The policy has the information and doesn't learn to use it: its loss is mostly motion, and choosing the cube is a small share of it.

### wrap-up — the word reaches the model; training doesn't bind it

**Buys:** a VAE that keeps colour (97-99% of cube pixels; was 0-7% for green and blue), and a policy that does everything in T5 except choose the cube: side word perfect, grasps square.

**Fails:** the colour word decides the cube in only a few episodes; the policy falls back on an arbitrary default (red, or far).

**Next:**
1. a near/far head on the video side trained alongside the actions (the named cube's slot is in each recording's `instructions.json`; train-time only), to test whether the binding is the missing piece;
2. if it is, the general version: the future-video loss (roadmap box 07), which needs the same binding without a sim-only label;
3. more T5 demos (the probe went 89% → 100% from 135 to 1,000 scenes).